In [1]:
from fpl_project.scripts import config
from fpl_project.scripts.model_utils.trainer import set_seed
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from glum import GeneralizedLinearRegressor

pd.set_option('display.width', 1000)
pd.set_option('display.max_columns', 15)

In [2]:
set_seed(77)
if not config.FEATURES_ENGINEERING.exists():
    from fpl_project.scripts.data_utils.features_processing import build_and_save_features_engineering_data
    build_and_save_features_engineering_data()
data = pd.read_parquet(config.FEATURES_ENGINEERING)
data.total_points = data.total_points.clip(lower=0)

In [3]:
train_season, test_season = 2526, 2627
print(f"Train season: {train_season} (rows: {(data.season == train_season).sum()})")
print(f"Test season:  {test_season} (rows: {(data.season == test_season).sum()})")

Train season: 2526 (rows: 29338)
Test season:  2627 (rows: 4584)


In [4]:
played_mask = data["is_played"].astype(bool) if "is_played" in data.columns else data.total_points.notna()
train = data[(data.season == train_season) & played_mask].copy().reset_index(drop=True)
test = data[(data.season == test_season) & played_mask].copy().reset_index(drop=True)
predict_df = data[~played_mask].copy().reset_index(drop=True)

y_train = train["total_points"].reset_index(drop=True)
y_test = test["total_points"].reset_index(drop=True)

In [5]:
preds = pd.read_parquet(config.ML_PREDS_PATH)

preds_test_dict = {}
preds_next_dict = {}
for algo_name, algo_path in [("glm", config.PREDS_GLM), ("xgboost", config.PREDS_XGBOOST), ("mlp", config.PREDS_MLP), ("conv1d", config.PREDS_CONV1D)]:
    if algo_path.exists():
        df_algo = pd.read_parquet(algo_path)
        if algo_name in df_algo.columns:
            preds_test_dict[algo_name] = df_algo.loc[df_algo["is_played"], algo_name].clip(lower=0, upper=15).values
            preds_next_dict[algo_name] = df_algo.loc[~df_algo["is_played"], algo_name].clip(lower=0, upper=15).values

preds_test = pd.DataFrame(preds_test_dict)
preds_next_gw_models = pd.DataFrame(preds_next_dict)

print(f"Preds train shape: {preds.shape}")
print(f"Preds test shape:  {preds_test.shape}")

Preds train shape: (29338, 8)
Preds test shape:  (4258, 4)


In [6]:
model_cols = [c for c in ["glm", "xgboost", "mlp", "conv1d"] if c in preds.columns and c in preds_test.columns]

X_train = preds[model_cols].copy().reset_index(drop=True)
X_test = preds_test[model_cols].copy().reset_index(drop=True)

print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape}  | y_test shape:  {y_test.shape}")
print(f"Nulls in X_train: {X_train.isna().sum().sum()} | Nulls in X_test: {X_test.isna().sum().sum()}")
print(f"Cechy wejsciowe meta-modelu ({len(model_cols)}): {model_cols}")

X_train shape: (29338, 4) | y_train shape: (29338,)
X_test shape:  (4258, 4)  | y_test shape:  (4258,)
Nulls in X_train: 0 | Nulls in X_test: 0
Cechy wejsciowe meta-modelu (4): ['glm', 'xgboost', 'mlp', 'conv1d']


In [7]:
print("Korelacja predykcji modeli bazowych na zbiorze treningowym meta-modelu:")
print(X_train.corr().round(4))

Korelacja predykcji modeli bazowych na zbiorze treningowym meta-modelu:
            glm  xgboost     mlp  conv1d
glm      1.0000   0.9655  0.9733  0.8641
xgboost  0.9655   1.0000  0.9712  0.8657
mlp      0.9733   0.9712  1.0000  0.8730
conv1d   0.8641   0.8657  0.8730  1.0000


In [8]:
baseline_metrics = []
for col in model_cols:
    p_test_single = np.clip(X_test[col].values, 0, 15)
    baseline_metrics.append({
        "Model bazowy": col.upper(),
        "Test MSE": mean_squared_error(y_test, p_test_single),
        "Test MAE": mean_absolute_error(y_test, p_test_single),
        "Test R2": r2_score(y_test, p_test_single)
    })

baseline_df = pd.DataFrame(baseline_metrics).sort_values("Test MSE").reset_index(drop=True)
print("Wyniki pojedynczych modeli bazowych na zbiorze testowym:")
print(baseline_df)

Wyniki pojedynczych modeli bazowych na zbiorze testowym:
  Model bazowy  Test MSE  Test MAE   Test R2
0      XGBOOST  4.742982  1.184701  0.271109
1          GLM  4.844134  1.222758  0.255564
2          MLP  4.883539  1.191623  0.249509
3       CONV1D  5.379727  1.307869  0.173256


In [9]:
glm_meta = GeneralizedLinearRegressor(
    family="normal",
    alpha=0.01,
    l1_ratio=0.0,
    random_state=77
)
glm_meta.fit(X_train, y_train)

In [10]:
p_train_avg = np.clip(X_train.mean(axis=1).values, 0, 15)
p_test_avg = np.clip(X_test.mean(axis=1).values, 0, 15)

models = {
    "Simple Average (Baseline Ensemble)": None,
    "GLM (Ridge Normal Meta-Model)": glm_meta,
}

metrics_list = []
for name, model in models.items():
    if model is None:
        p_train = p_train_avg
        p_test = p_test_avg
    else:
        p_train = np.clip(model.predict(X_train), 0, 15)
        p_test = np.clip(model.predict(X_test), 0, 15)
    metrics_list.append({
        "Model": name,
        "Train MSE": mean_squared_error(y_train, p_train),
        "Train MAE": mean_absolute_error(y_train, p_train),
        "Train R2": r2_score(y_train, p_train),
        "Test MSE": mean_squared_error(y_test, p_test),
        "Test MAE": mean_absolute_error(y_test, p_test),
        "Test R2": r2_score(y_test, p_test)
    })

results_df = pd.DataFrame(metrics_list).sort_values("Test MSE").reset_index(drop=True)
print("Porownanie meta-modelu ze srednia prosta:")
print(results_df.to_string(index=False))

Porownanie meta-modelu ze srednia prosta:
                             Model  Train MSE  Train MAE  Train R2  Test MSE  Test MAE  Test R2
     GLM (Ridge Normal Meta-Model)   3.652951   0.985767  0.338072  4.680106  1.215561 0.280772
Simple Average (Baseline Ensemble)   3.735504   0.973395  0.323113  4.793007  1.199735 0.263421


In [11]:
glm_importance = pd.DataFrame({
    "model_feature": X_train.columns,
    "coef": glm_meta.coef_
}).sort_values(by="coef", key=abs, ascending=False).reset_index(drop=True)
print(f"GLM Intercept: {glm_meta.intercept_:.4f}")
print("Wspolczynniki wag modeli bazowych w GLM Meta-Model:")
print(glm_importance.to_string(index=False))

GLM Intercept: -0.0511
Wspolczynniki wag modeli bazowych w GLM Meta-Model:
model_feature     coef
      xgboost 0.634704
          mlp 0.239185
       conv1d 0.192545
          glm 0.079899


In [12]:
meta_cols = ["name", "code", "team", "opponent_team", "position", "was_home", "season", "gw", "kickoff_time"]
test_meta = test[meta_cols].copy().reset_index(drop=True)
test_meta["is_played"] = True

if not predict_df.empty and not preds_next_gw_models.empty:
    X_pred_meta = preds_next_gw_models[model_cols].copy().reset_index(drop=True)
    pred_meta = predict_df[meta_cols].copy().reset_index(drop=True)
    pred_meta["is_played"] = False
    season_2627_meta = pd.concat([test_meta, pred_meta], ignore_index=True)
else:
    X_pred_meta = None
    season_2627_meta = test_meta

out_df = season_2627_meta.copy()
p_test = np.clip(glm_meta.predict(X_test), 0, 15)
if X_pred_meta is not None:
    p_next = np.clip(glm_meta.predict(X_pred_meta), 0, 15)
    out_df["glm_meta"] = np.concatenate([p_test, p_next])
else:
    out_df["glm_meta"] = p_test

out_df.to_parquet(config.PREDS_GLM_META, index=False)
print(f"Saved glm_meta: {out_df.shape} -> {config.PREDS_GLM_META}")

Saved glm_meta: (4584, 11) -> C:\Users\russa\FPL-1\Data\preds_glm_meta.parquet
